<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 29 · Central-Bank and Macro Document Intelligence

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch29_central_bank_macro_document_intelligence.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

This chapter treats central-bank and macro documents as market data that must be handled with the same discipline as prices. You will learn to convert policy language into auditable tone fields, including stance, inflation concern, growth concern, and confidence, with a source snippet attached to every qualitative label. You will join extracted tone to curve reactions through explicit date and event-window fields, interpret two-year, ten-year, and slope changes in basis points, and keep the association descriptive rather than causal. You will also distinguish the synthetic teaching statements from the official FOMC document and curve-event snapshots stored in the companion data.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script reads two stored tables: extracted tone fields for three synthetic policy statements and the matching curve-reaction snapshot on the same dates. It prints the tone diagnostics and divides each two-year yield change by the extracted stance, so the front-end reaction per unit of stance can be compared across events. The three rows are illustrative; the official FOMC corpus in the data folder is a separate real snapshot.

In [ ]:
SCRIPT = ROOT / "code" / "ch29_central_bank_macro_document_intelligence.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Read the tone table as features, not facts: stances of +0.60, -0.15, and +0.90 come with extraction confidences of 0.88, 0.82, and 0.91.
Confirm the sign convention: positive stance means more restrictive language, and curve changes are two-year and ten-year moves in basis points.
Compare stance with the front end: +7, -5, and +11 bp two-year moves and slope changes of -3, +3, and -5 bp, so more restrictive language coincides with front-end increases and flattening in these three rows.
Treat front_end_per_stance (11.67, 33.33, 12.22) as a descriptive ratio; the near-zero stance of -0.15 makes the 33.33 figure unstable, so do not read precision into it.
Keep the synthetic rows separate from the official FOMC snapshots, and draw no causal conclusion from three observations.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch29_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Keyword tone scoring of two policy sentences

This lab isolates one mechanism: scoring policy tone by counting hawkish and dovish keyword hits in two mock policy sentences. The stance score is hawkish hits minus dovish hits divided by total hits, so it lies between -1 and +1, with positive values meaning more restrictive language. Both sentences are synthetic teaching text.

In [ ]:
sentences = {  # two mock policy sentences, not real statements
    "meeting_a": ("Inflation remains above target and policy stays "
                  "restrictive."),
    "meeting_b": ("Inflation pressures have eased and the committee "
                  "can proceed carefully."),
}
hawkish = ["above target", "restrictive", "tighten"]  # restrictive
dovish = ["eased", "carefully", "accommodative"]  # easing
for name, text in sentences.items():
    hawk = sum(text.count(word) for word in hawkish)
    dove = sum(text.count(word) for word in dovish)
    stance = (hawk - dove) / (hawk + dove)  # tone in [-1, 1]
    print(f"{name}: hawk {hawk} dove {dove} stance {stance:+.2f}")

### What to observe

Expected output: meeting_a has 2 hawkish and 0 dovish hits, stance +1.00; meeting_b has 0 hawkish and 2 dovish hits, stance -1.00. Keyword counting is transparent but coarse: the chapter's stored tone fields use graded scores such as +0.60 and -0.15, and real statements usually mix both tones in one paragraph.

## Exercises

1. **Sign convention.** State what a positive policy_stance means and how it should relate to the front end of the curve. *Hint:* Positive stance means more restrictive language; the two-year change is in basis points on the release-date observation.
2. **Join discipline.** Recompute the stance-to-curve link after excluding meeting_b and describe what changes. *Hint:* The stance of -0.15 is close to zero, which makes front_end_per_stance unstable at 33.33.
3. **Confounders.** List two confounders that could dominate the curve reaction on a release date. *Hint:* Positioning, prior expectations, and data surprises can outweigh the document signal.
4. **Official corpus.** Load ch29_fomc_documents_official.csv and confirm that it stores URLs, release dates, and text checksums alongside the official documents. *Hint:* The same-stem .meta.json file records the source institution and retrieval date.
5. **From count to field.** Replace the mini-lab keyword count with a three-level stance label (restrictive, neutral, easing) and justify each label with a source snippet. *Hint:* A tone label without a supporting snippet is a black-box opinion.

For more detail, see Chapter 29 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">